#  **Assignment 1 — Simplex Algorithm**
### **Piyush Anand**  
**Roll No:** CS25MTECH12009  

---

## **Assumptions**

The implementation of the Geometric Simplex Algorithm is based on the following assumptions:

1. **Polytope is non-degenerate**  
   Each vertex is formed by exactly `n` independent active constraints, ensuring a unique direction of movement from each vertex.

2. **Polytope is bounded**  
   The feasible region is closed and bounded, guaranteeing that an optimal solution exists and the algorithm terminates.

3. **Rank of A is n**  
   The constraint matrix `A` has full column rank, i.e., all active constraint vectors at a vertex are linearly independent.

4. **Initial feasible point is given**  
   The algorithm assumes that a valid initial basic feasible solution (`z₀`) is provided, which lies on `n` active constraints.

---

### **Objective**
To implement and trace the **Geometric (Vertex-to-Vertex) Simplex Algorithm**,  
showing each iteration, movement direction, ratio test, and the final optimal vertex.


In [ ]:
import numpy as np
import pandas as pd


## Reading Input Data

Each CSV file has the following format:

| Row | Description |
|------|--------------|
| 1 | Initial feasible vertex (`z₀`) |
| 2 | Cost vector (`c`) |
| 3 → end | Constraint matrix rows (`A`) and corresponding right-hand-side values (`b`) |

The function below reads the file and extracts these matrices and vectors.


In [ ]:
def read_input(filename):
    data = pd.read_csv(filename, header=None).values
    z0 = data[0, :-1]      # initial feasible vertex
    c  = data[1, :-1]      # cost vector
    A  = data[2:, :-1]     # constraint matrix
    b  = data[2:, -1]      # RHS
    m, n = A.shape
    return A, b, c, z0, m, n


##  Helper Functions for the Geometric Simplex

We now define smaller reusable functions that perform core mathematical steps:

- `identify_active_constraints()` → finds active constraints at current point  
- `compute_directions()` → computes edge directions from the current basis  
- `compute_cost_changes()` → evaluates objective change in each direction  
- `ratio_test()` → performs the ratio test to determine feasible step size  


In [ ]:
def identify_active_constraints(A, x0, b, tol=1e-9):
    """Find indices of constraints that are active (tight) at the current vertex."""
    return np.where(np.abs(A @ x0 - b) < tol)[0]


def compute_directions(A_B):
    """Compute edge directions from the active constraint matrix (basis)."""
    V = -np.linalg.inv(A_B)
    directions = [V[:, i] for i in range(A_B.shape[1])]
    return directions


def compute_cost_changes(c, directions):
    """Compute how much each direction improves the objective."""
    return [c @ v for v in directions]


def ratio_test(A, b, x, v, B, tol=1e-9):
    """
    Perform ratio test to find the step size (t*) and entering constraint index.
    Returns (t_star, entering_constraint) or (None, None) if unbounded.
    """
    m = len(b)
    t_candidates = []
    N = [i for i in range(m) if i not in B]

    print("t values for inactive constraints:")
    for j in N:
        a_j = A[j, :]
        denom = a_j @ v
        if denom > tol:
            t = (b[j] - a_j @ x) / denom
            if t > tol:
                t_candidates.append((t, j))
                print(f"      Constraint {j}: t = {t:.2f}")
            else:
                print(f"      Constraint {j}: t <= 0 → ignore")
        else:
            print(f"      Constraint {j}: denom <= 0 → ignore")

    if not t_candidates:
        return None, None  # unbounded
    return min(t_candidates)  # (t*, j_enter)


##  Simplex Algorithm

This function coordinates the simplex process:

1. Identify active constraints at the initial vertex.  
2. Compute directions and objective changes.  
3. Select the direction with the largest improvement (`cᵀv`).  
4. Run the ratio test to find the next constraint to become active.  
5. Update the basis (entering / leaving constraint).  
6. Repeat until no improving direction exists (optimal vertex found).  


In [ ]:
def geometric_simplex_trace(A, b, c, x0, tol=1e-9):
    """
    Simplex (vertex-to-vertex) algorithm.
    Assumes initial point x0 is a vertex (basic feasible solution).
    """
    m, n = A.shape

    # Step 1: Identify active constraints at the initial vertex
    active_constraints = identify_active_constraints(A, x0, b, tol)
    if len(active_constraints) < n:
        print("Error: Initial feasible point does not lie on enough active constraints.")
        return

    B = list(active_constraints[:n])
    iteration = 0
    print("Iter | Vertex (x)           | Obj    | Directions & t values")
    print("---------------------------------------------------------------")

    # Step 2: Main simplex loop
    while True:
        iteration += 1
        A_B = A[B, :]
        b_B = b[B]

        # Compute edge directions
        directions = compute_directions(A_B)
        cost_change = compute_cost_changes(c, directions)

        # Display current vertex
        print(f"{iteration:4d} | {np.round(x0,2)} | {c @ x0:7.2f} |")

        # Check for optimality
        if all(val <= tol for val in cost_change):
            print("\nReached optimal vertex.")
            print(f"x* = {np.round(x0,4)}, Objective = {c @ x0:.2f}")
            break

        # Select best improving direction
        i_enter = np.argmax(cost_change)
        v = directions[i_enter]
        print(f"  Moving along direction {i_enter} -> {np.round(v,2)}, c^T v = {cost_change[i_enter]:.4f}")

        # Ratio test to find step size and entering constraint
        t_star, j_enter = ratio_test(A, b, x0, v, B, tol)
        # if t_star is None:
        #     print("Problem is unbounded along this direction.")
        #     break

        # Move to new vertex
        x0 = x0 + t_star * v

        # Update basis (swap entering/leaving constraint)
        j_leave = B[i_enter]
        B[i_enter] = j_enter

        print(f"    -> Move t* = {t_star:.2f} → New vertex {np.round(x0,2)}")
        print(f"       Entering constraint: {j_enter}, Leaving constraint: {j_leave}")
        print("---------------------------------------------------------------")


In [ ]:
data = [
    [0, 5, ""],
    [3, 2, ""],
    [2, 1, 10],
    [1, 3, 15],
    [1, 0, 6],
    [-1, 0, 0],
    [0, -1, 0]
]

# Convert to DataFrame and save
df = pd.DataFrame(data)
df.to_csv("Testcase.csv", header=False, index=False)

A, b, c, z0, m, n = read_input("Testcase4.csv")
geometric_simplex_trace(A, b, c, z0)


Iter | Vertex (x)           | Obj    | Directions & t values
---------------------------------------------------------------
   1 | [0 0] |    0.00 |
  Moving along direction 1 -> [0. 1.], c^T v = 5.0000
t values for inactive constraints:
      Constraint 0: t = 6.00
      Constraint 1: t = 3.00
      Constraint 2: denom <= 0 → ignore
    -> Move t* = 3.00 → New vertex [0. 3.]
       Entering constraint: 1, Leaving constraint: 4
---------------------------------------------------------------
   2 | [0. 3.] |   15.00 |
  Moving along direction 0 -> [ 1.  -0.5], c^T v = 1.5000
t values for inactive constraints:
      Constraint 0: t = 2.00
      Constraint 2: t = 7.00
      Constraint 4: t = 6.00
    -> Move t* = 2.00 → New vertex [2. 2.]
       Entering constraint: 0, Leaving constraint: 3
---------------------------------------------------------------
   3 | [2. 2.] |   18.00 |

Reached optimal vertex.
x* = [2. 2.], Objective = 18.00


# Summary

- The notebook implements the **vertex-to-vertex simplex algorithm**.  
- It assumes the **starting point is a vertex**.  
- The algorithm:
  - Moves from vertex to vertex along edges of the feasible region.
  - Chooses directions that increase the objective value.
  - Uses a ratio test to maintain feasibility.
- The process stops when:
  - No direction improves the objective (→ optimal vertex), or  
  - The problem is unbounded along an improving direction.


## Testing the Geometric Simplex Algorithm

In this section, we’ll:
1. Generate a few **test case CSV files** automatically (2D linear programs).  
2. Use the `geometric_simplex_trace()` function to solve them.  
3. Observe the printed simplex trace, showing vertex updates and objective improvements.

Each test case has:
- Initial vertex `z₀`
- Cost vector `c`
- Constraint matrix `A` and RHS `b`

The CSV structure matches what the `read_input()` function expects.


In [ ]:
import pandas as pd
import numpy as np

def create_testcase(filename, z0, c, A, b):
    """Creates a CSV file compatible with the read_input() format."""
    data = []
    data.append(list(z0) + [""])         # z0 row
    data.append(list(c) + [""])          # cost vector row
    for i in range(len(A)):
        data.append(list(A[i]) + [b[i]]) # constraints with RHS
    df = pd.DataFrame(data)
    df.to_csv(filename, header=False, index=False)
    print(f"Created {filename}")


# ---------------------------------------------------------------------
# Testcase 1 — Multi-step 2D problem (3-vertex path)
# maximize z = 3x1 + 2x2
# subject to:
#     2x1 + x2 <= 10
#     x1 + 3x2 <= 15
#     x1 <= 6
#     x1, x2 >= 0
# Starting at z0 = (0, 5) — feasible but not optimal.
# ---------------------------------------------------------------------
z0 = np.array([0, 5])
c  = np.array([3, 2])
A  = np.array([[2, 1],
               [1, 3],
               [1, 0],
               [-1, 0],
               [0, -1]])
b  = np.array([10, 15, 6, 0, 0])
create_testcase("Testcase1.csv", z0, c, A, b)


# ---------------------------------------------------------------------
# Testcase 2 — 3D problem (requires several pivots)
# maximize z = 4x1 + 3x2 + 5x3
# subject to:
#     2x1 +  x2 +  x3 <= 10
#     x1 + 3x2 + 2x3 <= 15
#     x1 +  x2 + 4x3 <= 18
#     x1, x2, x3 >= 0
# Start at z0 = (0, 0, 0)
# ---------------------------------------------------------------------
z0 = np.array([0, 0, 0])
c  = np.array([4, 3, 5])
A  = np.array([[2, 1, 1],
               [1, 3, 2],
               [1, 1, 4],
               [-1, 0, 0],
               [0, -1, 0],
               [0, 0, -1]])
b  = np.array([10, 15, 18, 0, 0, 0])
create_testcase("Testcase2.csv", z0, c, A, b)


# ---------------------------------------------------------------------
#  Testcase 3 — 2D polygon with 4 constraints (2–3 pivots)
# maximize z = 5x1 + 2x2
# subject to:
#     x1 + x2 <= 8
#     3x1 + 2x2 <= 18
#     x1 <= 6
#     x2 <= 5
#     x1, x2 >= 0
# Start at z0 = (0, 5)
# ---------------------------------------------------------------------
z0 = np.array([0, 5])
c  = np.array([5, 2])
A  = np.array([[1, 1],
               [3, 2],
               [1, 0],
               [0, 1],
               [-1, 0],
               [0, -1]])
b  = np.array([8, 18, 6, 5, 0, 0])
create_testcase("Testcase3.csv", z0, c, A, b)




Created Testcase1.csv
Created Testcase2.csv
Created Testcase3.csv


## Run the Algorithm on the Test Cases

We now load each test case file and call `geometric_simplex_trace()`  
to print the vertex-by-vertex trace of the simplex algorithm.


In [ ]:
for i in range(1, 5):
    print(f"\n===== Running Testcase {i} =====")
    A, b, c, z0, m, n = read_input(f"Testcase{i}.csv")
    geometric_simplex_trace(A, b, c, z0)



===== Running Testcase 1 =====
Iter | Vertex (x)           | Obj    | Directions & t values
---------------------------------------------------------------
   1 | [0. 5.] |   10.00 |
  Moving along direction 1 -> [ 1.   -0.33], c^T v = 2.3333
t values for inactive constraints:
      Constraint 0: t = 3.00
      Constraint 2: t = 6.00
      Constraint 4: t = 15.00
    -> Move t* = 3.00 → New vertex [3. 4.]
       Entering constraint: 0, Leaving constraint: 3
---------------------------------------------------------------
   2 | [3. 4.] |   17.00 |

Reached optimal vertex.
x* = [3. 4.], Objective = 17.00

===== Running Testcase 2 =====
Iter | Vertex (x)           | Obj    | Directions & t values
---------------------------------------------------------------
   1 | [0. 0. 0.] |    0.00 |
  Moving along direction 2 -> [0. 0. 1.], c^T v = 5.0000
t values for inactive constraints:
      Constraint 0: t = 10.00
      Constraint 1: t = 7.50
      Constraint 2: t = 4.50
    -> Move t* = 4.50 